# Decay Timing Analysis
Correlates stable-phase metrics with probe decay final validation loss.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

In [ ]:
# Point this to your run directory
LOG_DIR = Path("../logs")  # adjust if running from a different location

# List available runs
runs = sorted(LOG_DIR.iterdir())
for i, r in enumerate(runs):
    has_sweep = (r / "decay_sweep.jsonl").exists()
    print(f"{i:2d}  {r.name}  {'[sweep]' if has_sweep else ''}")

In [ ]:
# Select a run
RUN = runs[-1]  # or pick by index / name
print("Using:", RUN.name)

def read_jsonl(path):
    with open(path) as f:
        return pd.DataFrame(json.loads(line) for line in f)

metrics = read_jsonl(RUN / "metrics.jsonl")
sweep   = read_jsonl(RUN / "decay_sweep.jsonl")

print(f"metrics rows: {len(metrics)}, sweep rows: {len(sweep)}")
sweep.head()

In [ ]:
# Training loss curve with probe timing markers
train_rows = metrics[metrics["train_loss"].notna()]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_rows["step"], train_rows["train_loss"], label="train loss")
for step in sweep["probe_start_step"]:
    ax.axvline(step, color="orange", alpha=0.4, linewidth=1)
ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.set_title("Training loss + probe decay timings")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Probe val loss vs decay start step
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sweep["probe_start_step"], sweep["probe_final_val_loss"], marker="o")
ax.set_xlabel("probe start step")
ax.set_ylabel("probe final val loss")
ax.set_title("Final val loss after probe decay")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix: stable-phase metrics vs probe val loss
METRIC_COLS = [
    "train_loss", "loss_variance", "loss_oscillation", "loss_improvement_rate",
    "grad_norm", "grad_snr", "grad_weight_ratio", "grad_cosine_sim",
    "adam_v_norm", "weight_norm", "param_update_norm",
]

# Keep only columns that exist
metric_cols = [c for c in METRIC_COLS if c in sweep.columns]
y = sweep["probe_final_val_loss"].values

rows = []
for col in metric_cols:
    x = sweep[col].values
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        continue
    pearson_r, pearson_p = stats.pearsonr(x[mask], y[mask])
    spearman_r, spearman_p = stats.spearmanr(x[mask], y[mask])
    rows.append({
        "metric": col,
        "pearson_r": pearson_r, "pearson_p": pearson_p,
        "spearman_r": spearman_r, "spearman_p": spearman_p,
    })

corr_df = pd.DataFrame(rows).sort_values("pearson_r", key=abs, ascending=False)
corr_df.style.background_gradient(subset=["pearson_r", "spearman_r"], cmap="RdBu_r", vmin=-1, vmax=1)

In [ ]:
# Scatter plots: each metric vs probe final val loss
n = len(metric_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

for ax, col in zip(axes, metric_cols):
    x = sweep[col].values
    ax.scatter(x, y, color="steelblue", edgecolors="white", linewidth=0.5)
    # trend line
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() >= 2:
        m, b = np.polyfit(x[mask], y[mask], 1)
        xline = np.linspace(x[mask].min(), x[mask].max(), 100)
        ax.plot(xline, m * xline + b, color="tomato", linewidth=1.5)
    row = corr_df[corr_df["metric"] == col]
    r = row["pearson_r"].values[0] if len(row) else float("nan")
    ax.set_title(f"{col}\nPearson r={r:.2f}")
    ax.set_xlabel(col)
    ax.set_ylabel("probe val loss")

for ax in axes[len(metric_cols):]:
    ax.set_visible(False)

plt.suptitle("Stable-phase metrics vs probe decay final val loss", y=1.01)
plt.tight_layout()
plt.show()